In [35]:
from openai import OpenAI
from dotenv import load_dotenv
import os
from pathlib import Path
import json
load_dotenv()

True

In [36]:
DATA = Path('data')

standards = (DATA / 'team_standards.md').read_text("utf-8")
payment_code = (DATA / 'payment_utils.py').read_text("utf-8")
auth_code = (DATA / 'auth.py').read_text("utf-8")

In [37]:
def call_llm(prompt):
    client = OpenAI()
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": prompt}
        ]
    )
    return response.choices[0].message.content

In [38]:
# print(standards)

prompt = f"""Review this Python code against our team standards.
Give 2-3 issues with line numbers. Do NOT claim you posted a PR comment.

STANDARDS:
{standards}

FILE : payment_utils.py

{payment_code}

"""

call_llm(prompt)

'Here are a few issues with the provided Python code in `payment_utils.py` against the team standards:\n\n1. **Line 6 (get_user function)**: The code uses string interpolation to create a SQL query, which is vulnerable to SQL injection attacks. According to the standards, all DB queries must use parameterized queries (e.g., SQLAlchemy). This line should use parameterized queries instead.\n\n   ```python\n   query = f"SELECT * FROM users WHERE id = {user_id}"\n   ```\n\n2. **Line 11 (process_payment function)**: The code logs the card number with the print statement, which violates the standard of not logging sensitive data such as card numbers. This line should be modified to avoid logging such information.\n\n   ```python\n   print(f"Processing {amount} for card {card_number}")\n   ```\n\n3. **Line 3 (process_payment function)**: The function lacks validation of the user input (`amount` and `card_number`) before processing the payment. According to the standards, all user inputs must 

In [39]:
print(call_llm(prompt))

Here are the issues identified in the provided code against the team standards:

1. **Line 5**: The query in the `get_user` function uses an unparameterized SQL query, which violates the standard that all DB queries must use parameterized queries (SQLAlchemy). The current implementation risks SQL injection.

   ```python
   query = f"SELECT * FROM users WHERE id = {user_id}"
   ```

2. **Line 8**: The `process_payment` function logs sensitive data by printing the card number, which violates the standard of never logging sensitive data.

   ```python
   print(f"Processing {amount} for card {card_number}")
   ```

3. **Line 12**: The `process_payment` function does not validate the user inputs (like `amount` and `card_number`) before processing the payment, violating the standard that all user inputs must be validated before DB queries.

   ```python
   def process_payment(amount, card_number):
   ```

These issues should be addressed to comply with the team's coding standards.


In [40]:
##tools
#allowed_files = ['payment_utils.py', 'auth.py']

pr = {
    "number": 247,
    "comments": [],
    "status": "PENDING",
}


def read_file(filename):
    return (DATA / filename).read_text("utf-8")

def post_review_comment(filename,comment):
    pr['comments'].append({
        "filename": filename,
        "comment": comment
    })
    return "Comment Saved"

def set_pr_status(status):
    if status not in ["PASS", "BLOCKED"]:
        return "ERROR: status must be either PASS or BLOCKED"
    pr['status'] = status
    return f"PR status updated"



In [41]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "read_file",
            "description": "Read the content of a file. Use this to read files that are part of the PR.",
            "parameters": {
                "type": "object",
                "properties": {
                    "filename": {
                        "type": "string"
                    }
                },
                "required": ["filename"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "post_review_comment",
            "description": "Post a review comment on the PR. Use this to provide feedback on the code.",
            "parameters": {
                "type": "object",
                "properties": {
                    "filename": {
                        "type": "string"
                    },
                    "comment": {
                        "type": "string"
                    }
                },
                "required": ["filename", "comment"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "set_pr_status",
            "description": "Set the status of the PR. Use this to indicate whether the PR is ready to be merged or not.",
            "parameters": {
                "type": "object",
                "properties": {
                    "status": {
                        "type": "string",
                        "enum": ["PASS", "BLOCKED"]
                    }
                },
                "required": ["status"]
            }
        }
    }
]


In [42]:
demo_prompt = [
    {"role": "user", "content": "Please read auth.py before reviewing it"}
]


client = OpenAI()

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=demo_prompt,
    tools=tools
)

response.choices[0].message.content


In [43]:
print(response)

ChatCompletion(id='chatcmpl-EfnnGNoPKxqCwJxFDnYT20lTCJAQY', choices=[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_f9yJe8mzg3vAyM3LBtQTThRn', function=Function(arguments='{"filename":"auth.py"}', name='read_file'), type='function')]))], created=1793631562, model='gpt-4o-mini-2024-07-18', object='chat.completion', metadata=None, moderation=None, service_tier='default', system_fingerprint='fp_2851888338', usage=CompletionUsage(completion_tokens=15, prompt_tokens=143, total_tokens=158, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0, text_tokens=None), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cache_write_tokens=None, cached_tokens=0, image_tokens=None, text_tokens=None)))


In [44]:
first_message = response.choices[0].message

In [45]:
#first_message.tool_calls[0].function.name
#first_message.tool_calls[0].function.arguments


In [46]:
call = first_message.tool_calls[0]

In [47]:
arguments = json.loads(first_message.tool_calls[0].function.arguments)

In [48]:
file_text = read_file(arguments['filename'])
file_text

'def login(username, password):\n    if username == "admin" and password == "admin123":\n        return True\n    return False\n\n\ndef generate_token(user_id):\n    import time\n    return f"{user_id}_{time.time()}"'

In [49]:
assistant_message = {
    "role": "assistant",
    "content": None,
    "tool_calls": [
        {
            "id": call.id,
            "type": "function",
            "function": {
                "name": call.function.name,
                "arguments": call.function.arguments,
            },
        }
    ],
}

demo_prompt.append(assistant_message)


In [50]:
demo_prompt

[{'role': 'user', 'content': 'Please read auth.py before reviewing it'},
 {'role': 'assistant',
  'content': None,
  'tool_calls': [{'id': 'call_f9yJe8mzg3vAyM3LBtQTThRn',
    'type': 'function',
    'function': {'name': 'read_file',
     'arguments': '{"filename":"auth.py"}'}}]}]

In [51]:
demo_prompt.append(
    {
        "role": "tool",
        "tool_call_id": call.id,
        "name": call.function.name,
        "content": file_text,
    }
)


In [52]:
response_se = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=demo_prompt,
    tools=tools
)

print(response_se.choices[0].message.content)


The `auth.py` file contains two functions:

1. **`login(username, password)`**:
   - This function checks if the provided username and password match the hardcoded credentials (`username` as "admin" and `password` as "admin123").
   - If the credentials match, it returns `True`; otherwise, it returns `False`.

2. **`generate_token(user_id)`**:
   - This function generates a token for a given `user_id` by combining the `user_id` and the current time (in seconds since the epoch). This token format can potentially be used for session management or authentication tokens.

### Review Comments:
- **Security Concerns**:
  - Hardcoding credentials (like "admin" and "admin123") is a security risk. It's better to use environment variables or a configuration system to manage sensitive information.
  - The `generate_token` function utilizes the current time, which might not be secure for token generation. Consider using cryptographic methods to generate secure tokens (e.g., JWT).

- **Functionalit

In [53]:
## Build the full agent (single agent) using python

In [67]:
functions = {
    "read_file": read_file,
    "post_review_comment": post_review_comment,
    "set_pr_status": set_pr_status
}



def run_agent(question, max_steps=10):

    # step 1 prepare the messages
    messages = [
        {
            "role": "system",
            "content": (
                "You are a code review agent. "
                "First read team_standards.md. "
                "Then review payment_utils.py, auth.py, and order.py. "
                "Post comments for confirmed important issues. "
                "Set PR status to BLOCKED if important violations exist, "
                "otherwise PASS. Finally, summarize the review."
            )
        },
        {
            "role": "user",
            "content": question
        }
    ]

    step = 0

    while step < max_steps:
        step += 1
        print(f"\n===== Agent Step {step} =====")

        # THINK : Ask LLM what to do next
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            tools=tools,
            tool_choice='auto'
        )

        message = response.choices[0].message
        # Remember the LLM's response
        messages.append(message)

        # STOP : NO more tools call
        if not message.tool_calls:
            print("agent finished")
            return message.content
        
        for call in  message.tool_calls:
            tool_name = call.function.name
            print(f"Calling tool: {tool_name}")

            try:
                arguments = json.loads(call.function.arguments)
                if tool_name not in functions:
                    raise ValueError(f"Unknown tool: {tool_name}")

                python_function = functions[tool_name]
                result = python_function(**arguments)

            except Exception as e:
                result = f"Tool error: {str(e)}"

            messages.append({
                    "role": "tool",
                    "tool_call_id": call.id,
                    "content": str(result)
                })
            
    return "Agent stopped : Maximum steps reached"

In [68]:
pr = {
    "number": 247,
    "comments": [],
    "status": "PENDING",
}

In [69]:
pr["comments"] = []
pr["status"] = "PENDING"

answer = run_agent("Revew PR #247 using our team coding standards.")


===== Agent Step 1 =====
Calling tool: read_file

===== Agent Step 2 =====
Calling tool: read_file
Calling tool: read_file
Calling tool: read_file

===== Agent Step 3 =====
Calling tool: post_review_comment
Calling tool: post_review_comment
Calling tool: post_review_comment
Calling tool: set_pr_status

===== Agent Step 4 =====
agent finished


In [70]:
print(answer)

### Review Summary for PR #247

The review has identified several important violations of our team's coding standards:

1. **SQL Injection Vulnerability**: The `get_user` function in `payment_utils.py` uses direct string interpolation in SQL queries, which exposes the application to SQL injection risks. It should utilize parameterized queries with SQLAlchemy instead.
   
2. **Logging of Sensitive Data**: The `process_payment` function in `payment_utils.py` prints the card number, which violates our standard of never logging sensitive data.

3. **JWT Standards Compliance**: The `generate_token` function in `auth.py` does not produce a JWT token according to the defined standards, specifically missing an expiry.

Due to these issues, the PR status has been set to **BLOCKED**. 

Please address these violations before resubmitting the PR.


In [71]:
pr


{'number': 247,
 'comments': [{'filename': 'payment_utils.py',
   'comment': 'DB querying in get_user function is using a direct string interpolation which is vulnerable to SQL injection. Use parameterized queries with SQLAlchemy.'},
  {'filename': 'payment_utils.py',
   'comment': 'Sensitive data such as card numbers should not be logged or printed. Avoid logging sensitive information.'},
  {'filename': 'auth.py',
   'comment': 'The token generated in generate_token does not follow JWT standards. JWT tokens must include expiry.'}],
 'status': 'BLOCKED'}